In [1]:
import librosa
import numpy as np
from spafe.features.lfcc import lfcc
from spafe.utils.preprocessing import SlidingWindow

In [8]:
def extract_lfcc(
    wav_path,
    sr=16000,
    num_ceps=20,
    nfilts=46,
    nfft=400,        # 25 ms @16k
    win_len=0.025,
    win_hop=0.01
):
    signal, sr = librosa.load(wav_path, sr=sr)
    window = SlidingWindow(
        win_len=win_len,
        win_hop=win_hop,
        win_type="hamming"
    )
    lfcc_feat = lfcc(
        sig=signal,
        fs=sr,
        num_ceps=num_ceps,
        nfilts=nfilts,
        nfft=nfft,
        window=window
    )
    lfcc_delta = librosa.feature.delta(lfcc_feat)

    # delta-delta
    lfcc_delta2 = librosa.feature.delta(lfcc_feat, order=2)

    # ghép → LFCC-60
    lfcc_60 = np.hstack([lfcc_feat, lfcc_delta, lfcc_delta2])
    
    return lfcc_60   # shape: (num_frames, num_ceps)


In [9]:
def build_lfcc_dataset(
    wav_list,
    sr=16000,
    out_path="X_mfcc.npy"
):
    all_feats = []

    for wav_path in wav_list:
        print(wav_path)
        lfcc = extract_lfcc(wav_path, sr)
        all_feats.append(lfcc)

        print(f"{wav_path} -> {lfcc.shape}")

    # Gộp theo frame
    X = np.vstack(all_feats)
    X = X[:int(X.shape[0]//10), :]
    np.save(out_path, X)
    print("Saved:", out_path, X.shape)

    return X


In [4]:
# n_sp_m_feature = extract_lfcc("C:/Users/Vinh/Documents/Pythonfile/Voice-Activity-Detect/data/test/non_speech_mix.wav")
# n_sp_feature = extract_lfcc("C:/Users/Vinh/Documents/Pythonfile/Voice-Activity-Detect/data/test/non_speech.wav")
# sp_m_feature = extract_lfcc('C:/Users/Vinh/Documents/Pythonfile/Voice-Activity-Detect/data/test/speech_mix.wav')
# sp_feature = extract_lfcc('C:/Users/Vinh/Documents/Pythonfile/Voice-Activity-Detect/data/test/speech.wav')


# import matplotlib.pyplot as plt

# mfcc_list = [n_sp_m_feature, n_sp_feature, sp_m_feature, sp_feature]
# titles = ['Non-speech Mixed', 'Non-speech', 'Speech Mixed', 'Speech']

# plt.figure(figsize=(12, 8))

# for i, mfcc in enumerate(mfcc_list):
#     plt.subplot(4, 1, i + 1)
#     plt.imshow(mfcc.T, aspect='auto', origin='lower')
#     plt.colorbar(label='LFCC value')
#     plt.ylabel('LFCC coeff')
#     plt.title(titles[i])

# plt.xlabel('Frame index')
# plt.tight_layout()
# plt.show()



In [12]:
DIR = "D:/Pythonfile/Voice-Activity-Detect/data/processed/non-speech/non-mixed/noise"

wav_list = [f"{DIR}/{i + 1}.wav" for i in range(926)]
wav_list.pop(766)

X_mfcc = build_lfcc_dataset(wav_list, out_path="D:/Pythonfile/Voice-Activity-Detect/data/feature/train/mini_lfcc_non-speech_2")


D:/Pythonfile/Voice-Activity-Detect/data/processed/non-speech/non-mixed/noise/1.wav
D:/Pythonfile/Voice-Activity-Detect/data/processed/non-speech/non-mixed/noise/1.wav -> (1316, 60)
D:/Pythonfile/Voice-Activity-Detect/data/processed/non-speech/non-mixed/noise/2.wav
D:/Pythonfile/Voice-Activity-Detect/data/processed/non-speech/non-mixed/noise/2.wav -> (4039, 60)
D:/Pythonfile/Voice-Activity-Detect/data/processed/non-speech/non-mixed/noise/3.wav
D:/Pythonfile/Voice-Activity-Detect/data/processed/non-speech/non-mixed/noise/3.wav -> (6403, 60)
D:/Pythonfile/Voice-Activity-Detect/data/processed/non-speech/non-mixed/noise/4.wav
D:/Pythonfile/Voice-Activity-Detect/data/processed/non-speech/non-mixed/noise/4.wav -> (1421, 60)
D:/Pythonfile/Voice-Activity-Detect/data/processed/non-speech/non-mixed/noise/5.wav
D:/Pythonfile/Voice-Activity-Detect/data/processed/non-speech/non-mixed/noise/5.wav -> (248, 60)
D:/Pythonfile/Voice-Activity-Detect/data/processed/non-speech/non-mixed/noise/6.wav
D:/Pyth